In [ ]:
import stormpy

path = "./engagement.pm"
prism_program = stormpy.parse_prism_program(path)
model = stormpy.build_model(prism_program)

print(model)
# --------------------------------------------------------------
# Model type: 	DTMC (sparse)
# States: 	5
# Transitions: 	12
# Reward Models:  none
# State Labels: 	9 labels
#    * deadlock -> 0 item(s)
#    * disengaged -> 1 item(s)
#    * success -> 1 item(s)
#    * converted -> 1 item(s)
#    * init -> 1 item(s)
#    * failure -> 1 item(s)
#    * engaged -> 1 item(s)
#    * browsing -> 1 item(s)
#    * abandoned -> 1 item(s)
# Choice Labels: 	none
# --------------------------------------------------------------

-------------------------------------------------------------- 
Model type: 	DTMC (sparse)
States: 	5
Transitions: 	12
Reward Models:  none
State Labels: 	9 labels
   * deadlock -> 0 item(s)
   * disengaged -> 1 item(s)
   * success -> 1 item(s)
   * converted -> 1 item(s)
   * init -> 1 item(s)
   * failure -> 1 item(s)
   * engaged -> 1 item(s)
   * browsing -> 1 item(s)
   * abandoned -> 1 item(s)
Choice Labels: 	none
-------------------------------------------------------------- 



# Model checking DTMC with StormPy

## Reachability

In [9]:
# Compiling the reachability spec.
# The spec asks, "from a given state, what is the probability of eventually reaching converted?""
formula_str = """P=? [ F "converted" ]"""
properties = stormpy.parse_properties(formula_str, prism_program)

In [ ]:
# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

# Perform model checking for the reachability property.
result = stormpy.model_checking(model, properties[0])
assert result.result_for_all_states
values = result.get_values()

initial = set(model.initial_states)

print(f"{'idx':>3}  {'labels':<20}  {'P(F "converted")':>16}")
print("-" * 54)
for state in model.states:
    names = ", ".join(sorted(model.labeling.get_labels_of_state(state.id)))
    print(f"{state.id:>3}  {names:<20}  {values[state.id]:>16.4f}")

# idx  labels                P(F "converted")
# ------------------------------------------------------
#   0  browsing, init                  0.5184
#   1  engaged                         0.6720
#   2  disengaged                      0.2400
#   3  abandoned, failure              0.0000
#   4  converted, success              1.0000


idx  labels                P(F "converted")
------------------------------------------------------
  0  browsing, init                  0.5184
  1  engaged                         0.6720
  2  disengaged                      0.2400
  3  abandoned, failure              0.0000
  4  converted, success              1.0000


In [12]:
# Bounded reachability property. 
# It asks, "from a given state, what is the probability of reaching converted within 10 steps?"
formula_bounded = """P=? [ F<=10 "converted" ]"""
properties_bounded = stormpy.parse_properties(formula_bounded, prism_program)

In [13]:
# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

# Perform model checking for the reachability property.
result = stormpy.model_checking(model, properties[0])
assert result.result_for_all_states
values = result.get_values()

initial = set(model.initial_states)

print(f"{'idx':>3}  {'labels':<20}  {'P(F "converted")':>16}")
print("-" * 54)
for state in model.states:
    names = ", ".join(sorted(model.labeling.get_labels_of_state(state.id)))
    print(f"{state.id:>3}  {names:<20}  {values[state.id]:>16.4f}")

idx  labels                P(F "converted")
------------------------------------------------------
  0  browsing, init                  0.5184
  1  engaged                         0.6720
  2  disengaged                      0.2400
  3  abandoned, failure              0.0000
  4  converted, success              1.0000


## Invariance

In [ ]:
# Invariance. Probability of entering abandoned (failure state)
formula_invariance = """P=? [ G !"abandoned" ]"""
properties_invariance = stormpy.parse_properties(formula_invariance, prism_program)

In [ ]:
# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties_invariance])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

# Perform model checking for the reachability property.
result = stormpy.model_checking(model, properties_invariance[0])
assert result.result_for_all_states

# Get the values for the invariance property.
values_invariance = result.get_values()

initial = set(model.initial_states)

print(f"{'idx':>3}  {'labels':<20}  {'P(G !\"abandoned\")':>16}")
print("-" * 54)
for state in model.states:
    names = ", ".join(sorted(model.labeling.get_labels_of_state(state.id)))
    print(f"{state.id:>3}  {names:<20}  {values_invariance[state.id]:>16.4f}")

# idx  labels                P(G !"abandoned")
# ------------------------------------------------------
#   0  browsing, init                  0.5184
#   1  engaged                         0.6720
#   2  disengaged                      0.2400
#   3  abandoned, failure              0.0000
#   4  converted, success              1.0000    


idx  labels                P(G !"abandoned")
------------------------------------------------------
  0  browsing, init                  0.5184
  1  engaged                         0.6720
  2  disengaged                      0.2400
  3  abandoned, failure              0.0000
  4  converted, success              1.0000


In [19]:
# Combining safety and reachability
# Tests the user not going to the failure state without being engaged.
formula_safety_reachability = """P=? [ !"abandoned" U "engaged" ]"""
properties_safety_reachability = stormpy.parse_properties(formula_safety_reachability, prism_program)

In [ ]:
# Repeat the model building and checking process for the new property.
options = stormpy.BuilderOptions([p.raw_formula for p in properties_safety_reachability])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

# Perform model checking for the reachability property.
result = stormpy.model_checking(model, properties_safety_reachability[0])
assert result.result_for_all_states

# Get the values for the safety and reachability property.
values_safety_reachability = result.get_values()

initial = set(model.initial_states)

print(f"{'idx':>3}  {'labels':<20}  {'P(!\"abandoned\" U \"engaged\")':>16}")
print("-" * 54)
for state in model.states:
    names = ", ".join(sorted(model.labeling.get_labels_of_state(state.id)))
    print(f"{state.id:>3}  {names:<20}  {values_safety_reachability[state.id]:>16.4f}")

# idx  labels                P(!"abandoned" U "engaged")
# ------------------------------------------------------
#   0  browsing, init                  0.7714
#   1  engaged                         1.0000
#   2  disengaged                      0.3571
#   3  abandoned, failure              0.0000
#   4  converted, success              0.0000

idx  labels                P(!"abandoned" U "engaged")
------------------------------------------------------
  0  browsing, init                  0.7714
  1  engaged                         1.0000
  2  disengaged                      0.3571
  3  abandoned, failure              0.0000
  4  converted, success              0.0000


## Verification

In [ ]:
# // Is conversion probability at least 50%?
# P>=0.5 [ F "converted" ]

# // Can we guarantee staying engaged for 5 steps with >30% probability?
# P>=0.3 [ G<=5 "engaged" ]

# // Is it certain we eventually reach a terminal state?
# P>=1 [ F ("converted" | "abandoned") ]

In [21]:
# Storm does not parse G<=5 directly, so use G<=k p == !(F<=k !p).
verification_specs = [
    (
        "Conversion probability is at least 50%",
        'P>=0.5 [ F "converted" ]',
    ),
    (
        "Stay engaged for 5 steps with at least 30% probability",
        'P>=0.3 [ !(F<=5 !("engaged")) ]',
    ),
    (
        "Eventually reach a terminal state with certainty",
        'P>=1 [ F ("converted" | "abandoned") ]',
    ),
]

initial_state = next(iter(model.initial_states))
for description, formula in verification_specs:
    prop = stormpy.parse_properties(formula, prism_program)[0]
    result = stormpy.model_checking(model, prop)
    print(f"{str(result.at(initial_state)):<5}  {description}")

True   Conversion probability is at least 50%
False  Stay engaged for 5 steps with at least 30% probability
True   Eventually reach a terminal state with certainty
